# M4. 벡터DB 적재 (04_m4_벡터DB적재)

산재 사고사례 6,032건을 임베딩해서 ChromaDB에 적재하고, 샘플 질문으로 검색을 확인합니다.

| 체크리스트 | 이 노트북의 위치 |
|---|---|
| M4-1 RAG에 사용할 컬럼 결정 | 1절 |
| M4-2 문서 구성과 chunk 기준 결정 | 1절 |
| M4-3 임베딩 생성 | 3절 |
| M4-4 ChromaDB 적재 | 3절 |
| M4-5 샘플 질문으로 검색 확인 | 4절 |

**사전 준비**: 프로젝트 폴더에 `.env` 파일을 두고 `OPENAI_API_KEY=키값`을 한 줄로 적어 둡니다. (따옴표·공백 없이, 키는 Git에 올리지 않습니다.)

이미 적재가 끝난 DB가 있으면 임베딩을 다시 만들지 않고 빠진 문서만 이어서 적재합니다.

## 필요 패키지
이 노트북은 **pandas, chromadb, openai, python-dotenv** 를 사용합니다. 프로젝트 환경(uv)에 설치되어 있어야 합니다.
`ModuleNotFoundError`가 나면 터미널에서 아래처럼 설치한 뒤 커널을 재시작하세요.
```
uv add pandas chromadb openai python-dotenv
```

## 0. 준비

In [ ]:
import os
from pathlib import Path
import pandas as pd
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

# data/processed가 있는 폴더를 위로 올라가며 찾는다
ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
P = ROOT / "data" / "processed"

load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), f"{ROOT / '.env'} 에 OPENAI_API_KEY=키값 을 넣고 커널을 재시작하세요."
client = OpenAI(timeout=30, max_retries=2)

EMBED_MODEL = "text-embedding-3-small"
DB_PATH = ROOT / "data" / "chroma"      # .gitignore 대상(재생성 가능)
COLLECTION = "sif"
print("프로젝트 폴더:", ROOT)

## 1. 문서 설계 (M4-1, M4-2)
- **사례 1건 = 문서 1개**. 평균 글자수가 약 370자로 짧아 chunk로 나누지 않습니다.
- 문서 본문에 넣는 컬럼: 업종 분류, 고위험작업(제조업등) 또는 재해종류(건설업), 재해개요, 기인물, 재해유발요인, 예방대책
- 메타데이터(필터 검색용): 업종구분, 중분류, 기인물, 재해종류, 재해연도
- 제조업등과 건설업은 컬럼 구조가 달라 업종별로 문서 머리말을 다르게 만듭니다.

In [ ]:
def read_processed(prefix):
    hits = sorted(P.glob(f"{prefix}*.csv"))
    assert hits, f"'{prefix}'로 시작하는 processed 파일이 없습니다. 02 노트북을 먼저 실행하세요."
    return pd.read_csv(hits[0], encoding="utf-8-sig")

mf, cs = read_processed("1_SIF_제조업등"), read_processed("1_SIF_건설업")

def val(r, k):
    """컬럼이 없거나 비어 있으면 빈 문자열."""
    return "" if (k not in r.index or pd.isna(r[k])) else str(r[k]).strip()

def build_doc(r, kind):
    if kind == "제조업등":
        head = [f"[업종] {val(r,'산재업종(대분류)')} > {val(r,'산재업종(중분류)')} > {val(r,'산재업종(소분류)')}",
                f"[고위험작업] {val(r,'고위험작업·상황')}"]
    else:
        head = [f"[업종] 건설업 > {val(r,'공종')} > {val(r,'작업명')} > {val(r,'단위작업명')}",
                f"[재해종류] {val(r,'재해종류')}"]
    body = [f"[재해개요] {val(r,'재해개요')}", f"[기인물] {val(r,'기인물')}",
            f"[재해유발요인] {val(r,'재해유발요인')}", f"[예방대책] {val(r,'위험성감소대책(예시)')}"]
    return "\n".join(x for x in head + body if not x.endswith("] "))

ids, docs, metas = [], [], []
for kind, d in [("제조업등", mf), ("건설업", cs)]:
    for i, r in d.iterrows():
        ids.append(f"{kind}-{i}")
        docs.append(build_doc(r, kind))
        metas.append({
            "업종구분": kind,
            "중분류": val(r, "산재업종(중분류)"),
            "기인물": val(r, "기인물_표준"),
            "재해종류": val(r, "재해종류"),
            "재해연도": int(r["재해연도"]) if pd.notna(r["재해연도"]) else 0,
        })

lens = pd.Series([len(x) for x in docs])
print(f"문서 수: {len(docs):,} | 글자수 평균 {lens.mean():.0f} / 최대 {lens.max()}")
assert len(set(ids)) == len(ids), "문서 id가 중복됩니다."

In [ ]:
# 문서 예시 (업종별 1건)
for kind, start in [("제조업등", 0), ("건설업", len(mf))]:
    print(f"----- {kind} 예시 -----")
    print(docs[start])
    print(metas[start], "\n")

## 2. 컬렉션 열기
코사인 유사도(`hnsw:space = cosine`)로 `sif` 컬렉션을 엽니다. 이미 있으면 그대로 이어서 씁니다.

In [ ]:
chroma = chromadb.PersistentClient(path=str(DB_PATH))
col = chroma.get_or_create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})
print("현재 적재된 문서:", col.count(), "/", len(docs))

## 3. 임베딩 생성과 적재 (M4-3, M4-4)
아직 없는 문서만 50개씩 임베딩해서 적재합니다. 중간에 멈춰도 다시 실행하면 이어집니다.
(`text-embedding-3-small`, 문서당 6,000자까지 사용)

In [ ]:
import time

def embed(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=[t[:6000] for t in texts])
    return [x.embedding for x in r.data]

have = set(col.get(include=[])["ids"]) if col.count() else set()
todo = [i for i, _id in enumerate(ids) if _id not in have]
print(f"새로 적재할 문서: {len(todo):,}건 (기존 {len(have):,}건은 건너뜀)")

B = 50
t0 = time.time()
for s in range(0, len(todo), B):
    batch = todo[s:s + B]
    col.add(ids=[ids[i] for i in batch],
            documents=[docs[i] for i in batch],
            metadatas=[metas[i] for i in batch],
            embeddings=embed([docs[i] for i in batch]))
    if (s // B) % 10 == 0:
        print(f"  적재 {min(s + B, len(todo)):,}/{len(todo):,}  ({time.time() - t0:.0f}초)")

print("완료. 컬렉션 문서 수:", col.count())
assert col.count() == len(docs), "적재 수가 문서 수와 다릅니다. 이 셀을 다시 실행하세요."

## 4. 샘플 질문으로 검색 확인 (M4-5)
질문을 임베딩해서 가장 가까운 사례 3건을 찾습니다. 점수는 `1 - 코사인 거리`(클수록 비슷)입니다.

In [ ]:
def search(q, k=3, where=None):
    res = col.query(query_embeddings=embed([q]), n_results=k, where=where)
    for doc, m, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        body = doc.split("[재해개요]")[-1][:70].replace("\n", " ")
        print(f"{1 - dist:.3f} | {m['업종구분']} | {m['중분류']} | {m['기인물']} | {body}")

print("Q. 지게차 작업 중 협착 사고 원인과 대책")
search("지게차 작업 중 협착 사고 원인과 대책")
print("\nQ. 비계에서 추락한 사고 (업종구분=건설업 필터)")
search("비계에서 추락한 사고", where={"업종구분": "건설업"})
print("\nQ. 컨베이어에 끼인 사고 (재해연도 2020 이후 필터)")
search("컨베이어에 끼인 사고", where={"재해연도": {"$gte": 2020}})

## 정리
- 6,032건(제조업등 2,573 + 건설업 3,459)이 `data/chroma`의 `sif` 컬렉션에 적재되었습니다.
- 메타데이터 필터(`업종구분`, `재해연도` 등)로 검색 범위를 좁힐 수 있습니다.
- 다음 단계(M5): 검색한 사례를 근거로 LLM이 답변을 만드는 RAG 체인 → `05_m5_RAG체인.ipynb`